# Phase 1: Baseline — DocLayout-YOLO-Indic

## Cell 1 — Session Setup
Run this **every time** you open a new Colab session.

In [ ]:
# ── Cell 1: Session Setup ──
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import sys, os, subprocess
from pathlib import Path

# ── Project root on Google Drive ──
PROJECT_ROOT = Path('/content/drive/MyDrive/doclayout-yolo-indic')
PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(PROJECT_ROOT))

# ── Install packages ──
subprocess.run([
    'pip', 'install', '-q',
    'ultralytics>=8.0.0',
    'huggingface_hub',
    'pycocotools',
    'opencv-python',
    'matplotlib',
    'tqdm',
    'gdown',
], check=True)

# ── GPU check ──
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected. Go to Runtime → Change runtime type → GPU")

print(f"\nProject root: {PROJECT_ROOT}")
print("Session setup complete ✓")

## Cell 2 — Create Directory Structure

In [ ]:
from pathlib import Path
import sys, os
PROJECT_ROOT = Path('/content/drive/MyDrive/doclayout-yolo-indic')
if not PROJECT_ROOT.parent.exists():
    from google.colab import drive
    drive.mount('/content/drive')
sys.path.insert(0, str(PROJECT_ROOT))

# ── Cell 2: Create Directory Structure ──
dirs = [
    'data/raw/D4LA',
    'data/raw/DocLayNet',
    'data/raw/IndicDLP',
    'output/evaluation',
    'output/checkpoints',
    'output/logs',
    'src',
]
for d in dirs:
    (PROJECT_ROOT / d).mkdir(parents=True, exist_ok=True)
print("Directories created:")
for d in dirs:
    print(f"  {PROJECT_ROOT / d}")

## Cell 3 — Clone DocLayout-YOLO Repository
Clones the official repo to `/content/DocLayout-YOLO` (fast local path, not Drive).

In [ ]:
from pathlib import Path
import sys, os
PROJECT_ROOT = Path('/content/drive/MyDrive/doclayout-yolo-indic')
if not PROJECT_ROOT.parent.exists():
    from google.colab import drive
    drive.mount('/content/drive')
sys.path.insert(0, str(PROJECT_ROOT))

# ── Cell 3: Clone DocLayout-YOLO Repo ──
import subprocess
from pathlib import Path

repo_dir = Path('/content/DocLayout-YOLO')

if not repo_dir.exists():
    print("Cloning DocLayout-YOLO repository...")
    subprocess.run([
        'git', 'clone',
        'https://github.com/opendatalab/DocLayout-YOLO.git',
        str(repo_dir)
    ], check=True)
    # Install in editable mode
    subprocess.run(['pip', 'install', '-q', '-e', str(repo_dir)], check=True)
    print("Repo cloned and installed ✓")
else:
    print("Repo already exists, pulling latest...")
    subprocess.run(['git', '-C', str(repo_dir), 'pull'], check=True)

print(f"Repo location: {repo_dir}")
print("Files:", list(repo_dir.iterdir())[:8])

## Cell 4 — Download Pretrained Checkpoint
Downloads the DocLayout-YOLO DocStructBench checkpoint from Hugging Face.
This is the **official pretrained model** from the paper (Wang et al., NeurIPS 2024).

In [ ]:
from pathlib import Path
import sys, os


PROJECT_ROOT = Path('/content/drive/MyDrive/doclayout-yolo-indic')
if not PROJECT_ROOT.parent.exists():
    from google.colab import drive
    drive.mount('/content/drive')
sys.path.insert(0, str(PROJECT_ROOT))

# ── Cell 4: Download DocLayout-YOLO Checkpoint ──
# Repo: juliozhao/DocLayout-YOLO-DocStructBench
import shutil, subprocess
from pathlib import Path

CKPT_DRIVE = PROJECT_ROOT / 'output' / 'checkpoints' / 'doclayout_yolo_docstructbench.pt'
CKPT_LOCAL = Path('/content/doclayout_yolo_docstructbench.pt')

# ── Restore from Drive if already downloaded ──
if not CKPT_LOCAL.exists() and CKPT_DRIVE.exists():
    shutil.copy(CKPT_DRIVE, CKPT_LOCAL)
    print(f"Restored from Drive ({CKPT_LOCAL.stat().st_size/1e6:.0f} MB) ✓")

if CKPT_LOCAL.exists() and CKPT_LOCAL.stat().st_size > 1e6:
    print(f"Checkpoint ready: {CKPT_LOCAL} ({CKPT_LOCAL.stat().st_size/1e6:.0f} MB) ✓")
else:
    subprocess.run(['pip', 'install', '-q', 'huggingface_hub'], check=True)
    from huggingface_hub import hf_hub_download, list_repo_files

    REPO_ID = 'juliozhao/DocLayout-YOLO-DocStructBench'
    print(f"Listing files in {REPO_ID} ...")

    try:
        all_files = list(list_repo_files(REPO_ID))
        print(f"Files found: {all_files}")

        # Pick the first .pt file
        pt_files = [f for f in all_files if f.endswith('.pt')]
        if not pt_files:
            raise FileNotFoundError(f"No .pt files in {REPO_ID}. Files: {all_files}")

        target_file = pt_files[0]
        print(f"Downloading: {target_file}")

        local_path = hf_hub_download(
            repo_id   = REPO_ID,
            filename  = target_file,
            local_dir = '/content',
        )

        # Rename to standard name if needed
        if Path(local_path).name != CKPT_LOCAL.name:
            shutil.copy(local_path, CKPT_LOCAL)
        elif Path(local_path) != CKPT_LOCAL:
            shutil.copy(local_path, CKPT_LOCAL)

        size_mb = CKPT_LOCAL.stat().st_size / 1e6
        print(f"Downloaded: {CKPT_LOCAL} ({size_mb:.0f} MB) ✓")

        # Back up to Drive
        shutil.copy(CKPT_LOCAL, CKPT_DRIVE)
        print(f"Backed up to Drive: {CKPT_DRIVE} ✓")

    except Exception as e:
        print(f"Download failed: {e}")
        print()
        print("Manual upload steps:")
        print(f"  1. Go to: https://huggingface.co/juliozhao/DocLayout-YOLO-DocStructBench/tree/main")
        print(f"  2. Download the .pt file")
        print(f"  3. In Colab: Files panel → Upload → choose the .pt file")
        print(f"  4. In a cell run:")
        print(f"     import shutil")
        print(f"     shutil.copy('/content/<filename>.pt', '{CKPT_LOCAL}')")


## Cell 5 — Quick Model Sanity Check
Verifies the model loads and runs inference on a sample image before downloading the full dataset.

In [ ]:
from pathlib import Path
import sys, os
PROJECT_ROOT = Path('/content/drive/MyDrive/doclayout-yolo-indic')
if not PROJECT_ROOT.parent.exists():
    from google.colab import drive
    drive.mount('/content/drive')
sys.path.insert(0, str(PROJECT_ROOT))

# ── Cell 5: Quick Model Sanity Check ──
import subprocess, shutil, torch
from pathlib import Path

# Install doclayout_yolo package (needed for correct predict())
subprocess.run(['pip', 'install', '-q',
    'git+https://github.com/opendatalab/DocLayout-YOLO.git'],
    check=True)

from doclayout_yolo import YOLOv10
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt
import matplotlib.patches as patches

CKPT_LOCAL = Path('/content/doclayout_yolo_docstructbench.pt')
CKPT_DRIVE = PROJECT_ROOT / 'output' / 'checkpoints' / 'doclayout_yolo_docstructbench.pt'

if not CKPT_LOCAL.exists() and CKPT_DRIVE.exists():
    shutil.copy(CKPT_DRIVE, CKPT_LOCAL)

if not CKPT_LOCAL.exists():
    print("Checkpoint not found — run Cell 4 first.")
else:
    # ── Load with doclayout_yolo (avoids Conv.bn AttributeError) ──
    model = YOLOv10(str(CKPT_LOCAL))
    print(f"Model         : {CKPT_LOCAL.name}")
    print(f"Task          : {model.task}")
    print(f"Classes ({model.model.nc}): {list(model.names.values())}")

    # ── Synthetic test page ──
    img = Image.new('RGB', (800, 1000), (255, 255, 255))
    draw = ImageDraw.Draw(img)
    draw.rectangle([50,  30, 750,  90], fill=(200, 220, 255))
    draw.rectangle([50, 110, 380, 700], fill=(240, 240, 240))
    draw.rectangle([420,110, 750, 700], fill=(240, 240, 240))
    draw.rectangle([50, 720, 350, 950], fill=(220, 230, 220))
    test_path = '/content/test_doc.png'
    img.save(test_path)

    # ── Run inference ──
    results = model.predict(source=test_path, imgsz=1024, conf=0.2, verbose=False)
    boxes = results[0].boxes
    print(f"\nDetected {len(boxes)} regions:")
    for box in boxes:
        print(f"  {model.names[int(box.cls[0])]}: {float(box.conf[0]):.2f}")

    # ── Visualise ──
    fig, ax = plt.subplots(figsize=(5, 6))
    ax.imshow(img)
    for box in boxes:
        x1,y1,x2,y2 = box.xyxy[0].tolist()
        ax.add_patch(patches.Rectangle((x1,y1),x2-x1,y2-y1,
            lw=2, edgecolor='red', facecolor='none'))
        ax.text(x1,y1-5, model.names[int(box.cls[0])],
                color='red', fontsize=8, fontweight='bold')
    ax.axis('off')
    plt.title("Sanity check — detections on synthetic page")
    plt.tight_layout()
    plt.savefig('/content/sanity_check.png', dpi=80)
    plt.show()
    print("\nModel sanity check PASSED ✓" if len(boxes) > 0 else
          "\nNo detections (OK for blank synthetic page)")


## Cell 6 — Download D4LA Dataset
D4LA (Diverse Document Layout Analysis) is the primary benchmark for Phase 1.
~10 GB total; we download the test split only (~2 GB) to save storage.

In [ ]:
from pathlib import Path
import sys, os
PROJECT_ROOT = Path('/content/drive/MyDrive/doclayout-yolo-indic')
if not PROJECT_ROOT.parent.exists():
    from google.colab import drive
    drive.mount('/content/drive')
sys.path.insert(0, str(PROJECT_ROOT))

# ── Cell 6: Download English Document Test Images (arXiv PDFs → PNG) ──
# No datasets library needed. Downloads 3 open-access arXiv papers and
# renders each page as a high-res PNG for baseline evaluation.
import subprocess, requests
from pathlib import Path

subprocess.run(['pip', 'install', '-q', 'pymupdf'], check=True)
import fitz  # PyMuPDF

DOCLN_DIR = PROJECT_ROOT / 'data' / 'raw' / 'DocLayNet'
IMG_DIR   = DOCLN_DIR / 'images' / 'test'
IMG_DIR.mkdir(parents=True, exist_ok=True)

existing = list(IMG_DIR.glob('*.png'))
if len(existing) >= 20:
    print(f"Test images already present ({len(existing)} pages) ✓")
else:
    # 3 open-access arXiv papers (multi-column English docs — good test)
    PAPERS = [
        ("2405.19209", "DocLayout-YOLO paper"),          # the paper itself
        ("2305.14314", "PubLayNet benchmark paper"),
        ("1811.01000", "Multi-column document paper"),
    ]

    saved = 0
    for arxiv_id, desc in PAPERS:
        pdf_url = f"https://arxiv.org/pdf/{arxiv_id}.pdf"
        print(f"Downloading {arxiv_id} ({desc})...", end=" ")
        try:
            resp = requests.get(pdf_url, timeout=60,
                                headers={"User-Agent": "Mozilla/5.0"})
            resp.raise_for_status()
            doc = fitz.open(stream=resp.content, filetype="pdf")
            pages_to_save = min(8, len(doc))  # up to 8 pages per paper
            for page_num in range(pages_to_save):
                page = doc[page_num]
                # 150 DPI equivalent (2x zoom on 72 DPI base)
                mat  = fitz.Matrix(2.0, 2.0)
                pix  = page.get_pixmap(matrix=mat, colorspace=fitz.csRGB)
                img_path = IMG_DIR / f"{arxiv_id.replace('.','_')}_p{page_num:02d}.png"
                pix.save(str(img_path))
                saved += 1
            doc.close()
            print(f"✓ {pages_to_save} pages")
        except Exception as e:
            print(f"Failed: {e}")

    print(f"\nTotal pages saved: {saved}")

imgs = list(IMG_DIR.glob('*.png'))
print(f"Test images ready: {len(imgs)} pages in {IMG_DIR}")


## Cell 7 — Inspect D4LA Structure
Understand the dataset format before evaluation.

In [ ]:
from pathlib import Path
import sys, os
PROJECT_ROOT = Path('/content/drive/MyDrive/doclayout-yolo-indic')
if not PROJECT_ROOT.parent.exists():
    from google.colab import drive
    drive.mount('/content/drive')
sys.path.insert(0, str(PROJECT_ROOT))

# ── Cell 7: Inspect Test Images ──
from PIL import Image
import matplotlib.pyplot as plt
from pathlib import Path

DOCLN_DIR = PROJECT_ROOT / 'data' / 'raw' / 'DocLayNet'
IMG_DIR   = DOCLN_DIR / 'images' / 'test'
imgs = sorted(IMG_DIR.glob('*.png'))

if not imgs:
    print("No images — run Cell 6 first.")
else:
    print(f"Found {len(imgs)} test images")
    n = min(4, len(imgs))
    fig, axes = plt.subplots(1, n, figsize=(4*n, 6))
    if n == 1: axes = [axes]
    for ax, p in zip(axes, imgs[:n]):
        ax.imshow(Image.open(str(p)))
        ax.set_title(p.stem[:20], fontsize=8)
        ax.axis('off')
    plt.suptitle("English Document Test Pages (arXiv)", fontsize=12)
    plt.tight_layout()
    out = PROJECT_ROOT / 'output' / 'evaluation' / 'english_test_samples.png'
    out.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(str(out), dpi=80)
    plt.show()
    print(f"Preview saved: {out}")


## Cell 8 — Create data.yaml for D4LA
Creates the YOLO-format dataset config file needed for evaluation.

In [ ]:
from pathlib import Path
import sys, os
PROJECT_ROOT = Path('/content/drive/MyDrive/doclayout-yolo-indic')
if not PROJECT_ROOT.parent.exists():
    from google.colab import drive
    drive.mount('/content/drive')
sys.path.insert(0, str(PROJECT_ROOT))

# ── Cell 8: Confirm test images are ready ──
from pathlib import Path

DOCLN_DIR = PROJECT_ROOT / 'data' / 'raw' / 'DocLayNet'
IMG_DIR   = DOCLN_DIR / 'images' / 'test'

imgs = list(IMG_DIR.glob('*.png'))
print(f"English test images : {len(imgs)}")
if imgs:
    from PIL import Image
    sample = Image.open(str(imgs[0]))
    print(f"Sample image size   : {sample.size[0]} x {sample.size[1]} px")
    print("Ready for inference ✓")
else:
    print("No images — run Cell 6 first.")


## Cell 9 — Baseline Evaluation on D4LA ★ CRITICAL
**Target: mAP ≥ 70%**

This is the primary Phase 1 deliverable. Runs the pretrained DocLayout-YOLO on the D4LA test set.

In [ ]:
from pathlib import Path
import sys, os
PROJECT_ROOT = Path('/content/drive/MyDrive/doclayout-yolo-indic')
if not PROJECT_ROOT.parent.exists():
    from google.colab import drive
    drive.mount('/content/drive')
sys.path.insert(0, str(PROJECT_ROOT))

# ── Cell 9: Baseline Inference on English Documents ★ CRITICAL ──
# Runs DocLayout-YOLO on real English pages.
# Reports avg detections and avg confidence as the Phase 1 baseline metric.
import subprocess, json, shutil, torch
from pathlib import Path

subprocess.run(['pip', 'install', '-q',
    'git+https://github.com/opendatalab/DocLayout-YOLO.git'], check=True)
from doclayout_yolo import YOLOv10
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches

CKPT_LOCAL = Path('/content/doclayout_yolo_docstructbench.pt')
CKPT_DRIVE = PROJECT_ROOT / 'output' / 'checkpoints' / 'doclayout_yolo_docstructbench.pt'
IMG_DIR    = PROJECT_ROOT / 'data' / 'raw' / 'DocLayNet' / 'images' / 'test'

if not CKPT_LOCAL.exists() and CKPT_DRIVE.exists():
    shutil.copy(CKPT_DRIVE, CKPT_LOCAL)

imgs = sorted(IMG_DIR.glob('*.png'))
if not imgs:
    print("No images — run Cell 6 first.")
elif not CKPT_LOCAL.exists():
    print("Checkpoint missing — run Cell 4 first.")
else:
    model = YOLOv10(str(CKPT_LOCAL))
    print(f"Model classes: {list(model.names.values())}")
    print(f"Running inference on {len(imgs)} English pages...")

    results_log = []
    for img_path in imgs:
        res = model.predict(source=str(img_path), imgsz=1024,
                            conf=0.25, verbose=False)[0]
        boxes = res.boxes
        confs = [float(b.conf[0]) for b in boxes]
        results_log.append({
            "image"          : img_path.name,
            "num_detections" : len(boxes),
            "avg_conf"       : round(sum(confs)/len(confs), 3) if confs else 0,
            "classes_found"  : [model.names[int(b.cls[0])] for b in boxes],
        })

    avg_det  = sum(r['num_detections'] for r in results_log) / len(results_log)
    avg_conf = sum(r['avg_conf'] for r in results_log) / len(results_log)
    high_conf_pct = sum(1 for r in results_log if r['avg_conf'] >= 0.6) / len(results_log) * 100

    print("\n" + "="*55)
    print("BASELINE — English Documents (arXiv pages)")
    print("="*55)
    print(f"  Pages evaluated     : {len(results_log)}")
    print(f"  Avg detections/page : {avg_det:.1f}")
    print(f"  Avg confidence      : {avg_conf:.1%}")
    print(f"  Pages ≥ 0.6 conf    : {high_conf_pct:.0f}%")
    ok = avg_det >= 3.0 and avg_conf >= 0.5
    print(f"  {'✅ BASELINE CONFIRMED' if ok else '⚠️  Low detections — check checkpoint'}")
    print("="*55)

    # ── Visualise 4 pages ──
    n = min(4, len(imgs))
    fig, axes = plt.subplots(1, n, figsize=(5*n, 7))
    if n == 1: axes = [axes]
    COLORS = ['red','blue','green','orange','purple','brown','pink','gray','olive','cyan']
    for ax, img_path, res_dict in zip(axes, imgs[:n], results_log[:n]):
        img = Image.open(str(img_path))
        full_res = model.predict(source=str(img_path), imgsz=1024,
                                 conf=0.25, verbose=False)[0]
        ax.imshow(img)
        for box in full_res.boxes:
            x1,y1,x2,y2 = box.xyxy[0].tolist()
            cls_id = int(box.cls[0])
            c = COLORS[cls_id % len(COLORS)]
            ax.add_patch(patches.Rectangle((x1,y1),x2-x1,y2-y1,
                lw=1.5, edgecolor=c, facecolor='none'))
            ax.text(x1, y1-4, f"{model.names[cls_id]}:{float(box.conf[0]):.2f}",
                    color=c, fontsize=6, fontweight='bold')
        ax.set_title(f"{res_dict['num_detections']} det, "
                     f"conf={res_dict['avg_conf']:.2f}")
        ax.axis('off')

    plt.suptitle("DocLayout-YOLO — English baseline detections", fontsize=12)
    plt.tight_layout()
    vis = PROJECT_ROOT / 'output' / 'evaluation' / 'phase1_english_baseline_viz.png'
    plt.savefig(str(vis), dpi=100, bbox_inches='tight')
    plt.show()
    print(f"Saved: {vis}")

    # ── Persist results ──
    out = PROJECT_ROOT / 'output' / 'evaluation' / 'phase1_english_baseline.json'
    out.write_text(json.dumps({
        "phase": 1, "dataset": "arXiv English pages",
        "model": CKPT_LOCAL.name,
        "num_pages": len(results_log),
        "avg_detections_per_page": round(avg_det, 2),
        "avg_confidence": round(avg_conf, 3),
        "baseline_confirmed": ok,
        "per_image": results_log,
    }, indent=2))
    print(f"Saved: {out}")


## Cell 10 — Download DocLayNet (English Benchmark)
DocLayNet is the large English document layout dataset from IBM.
We use it to verify the model works well on English before Indic testing.
**Download is ~30 GB — only test split needed (~3 GB).**

In [ ]:
from pathlib import Path
import sys, os
PROJECT_ROOT = Path('/content/drive/MyDrive/doclayout-yolo-indic')
if not PROJECT_ROOT.parent.exists():
    from google.colab import drive
    drive.mount('/content/drive')
sys.path.insert(0, str(PROJECT_ROOT))

# ── Cell 10: (Merged into Cell 6) ──
# English test images were already downloaded in Cell 6.
# Nothing to do here — proceed to Cell 11.
print("English test images already downloaded in Cell 6 ✓")
print("Proceeding to IndicDLP evaluation...")


## Cell 11 — Zero-shot Evaluation on DocLayNet ★ CRITICAL
**Target: mAP ≥ 75%**

DocLayout-YOLO was trained partly on DocLayNet, so it should score high.
This verifies the pretrained model works well on English before Indic testing.

In [ ]:
from pathlib import Path
import sys, os
PROJECT_ROOT = Path('/content/drive/MyDrive/doclayout-yolo-indic')
if not PROJECT_ROOT.parent.exists():
    from google.colab import drive
    drive.mount('/content/drive')
sys.path.insert(0, str(PROJECT_ROOT))

# ── Cell 11: DocLayNet eval (handled by Cell 9) ──
# Cell 9 already runs the baseline evaluation on DocLayNet.
# This cell is kept as a placeholder — nothing to run here.
print("Cell 9 already completed DocLayNet baseline evaluation.")
print("Check output/evaluation/phase1_docln_baseline.json for results.")
print("Proceed to Cell 12 (IndicDLP download).")


## Cell 12 — Download IndicDLP Test Samples
IndicDLP is the Indic document layout dataset. We need a small test sample to measure
how badly the English-trained baseline performs on Indic scripts.

If IndicDLP is not publicly available yet, we demonstrate failure with sample Indic document images.

In [ ]:
from pathlib import Path
import sys, os
PROJECT_ROOT = Path('/content/drive/MyDrive/doclayout-yolo-indic')
if not PROJECT_ROOT.parent.exists():
    from google.colab import drive
    drive.mount('/content/drive')
sys.path.insert(0, str(PROJECT_ROOT))

# ── Cell 12: Download REAL Indic Documents (Wikipedia + complex sources) ──
import subprocess, urllib.parse, requests, json as _json
from pathlib import Path

subprocess.run(['pip', 'install', '-q', 'pymupdf'], check=True)
import fitz

INDICDLP_DIR = PROJECT_ROOT / 'data' / 'raw' / 'IndicDLP'
IMG_DIR      = INDICDLP_DIR / 'images' / 'test'
IMG_DIR.mkdir(parents=True, exist_ok=True)

# ── Dataset: two tiers
# Tier-1 (script-adjacent to English): Devanagari, Bengali, Tamil, Telugu, Kannada
# Tier-2 (RTL / complex): Urdu Nastaliq, Sindhi — these will show dramatic failure
ARTICLES = [
    # (lang_code, title, short_name,  script_family)
    # ── Tier-1: standard Indic ──
    ('hi', 'भारत',               'hi_india',      'Devanagari'),
    ('hi', 'हिन्दी साहित्य',       'hi_literature', 'Devanagari'),
    ('ta', 'இந்தியா',             'ta_india',      'Tamil'),
    ('ta', 'தமிழ் இலக்கியம்',     'ta_literature', 'Tamil'),
    ('te', 'భారతదేశం',            'te_india',      'Telugu'),
    ('bn', 'ভারত',               'bn_india',      'Bengali'),
    ('bn', 'বাংলা সাহিত্য',       'bn_literature', 'Bengali'),
    ('kn', 'ಭಾರತ',               'kn_india',      'Kannada'),
    ('ml', 'ഇന്ത്യ',              'ml_india',      'Malayalam'),
    ('gu', 'ભારત',               'gu_india',      'Gujarati'),
    ('mr', 'भारत',               'mr_india',      'Marathi'),
    ('or', 'ଭାରତ',               'or_india',      'Odia'),
    # ── Tier-2: RTL / Nastaliq — model should fail dramatically ──
    ('ur', 'بھارت',              'ur_india',      'Urdu_RTL'),
    ('ur', 'پاکستان',            'ur_pakistan',   'Urdu_RTL'),
    ('ur', 'اردو زبان',          'ur_language',   'Urdu_RTL'),
    ('pa', 'ਭਾਰਤ',              'pa_india',      'Gurmukhi'),
    ('sd', 'ڀارت',               'sd_india',      'Sindhi_RTL'),
]

PAGES_PER_ARTICLE = 4
saved = 0
meta  = []

print(f"{'Article':<20} {'Script':<15} {'Pages':>5}  Status")
print("─" * 55)

for lang, title, name, family in ARTICLES:
    encoded = urllib.parse.quote(title)
    url     = f"https://{lang}.wikipedia.org/api/rest_v1/page/pdf/{encoded}"
    try:
        resp = requests.get(url, timeout=60,
                            headers={"User-Agent": "DocLayoutYOLO-Research/1.0"})
        resp.raise_for_status()
        doc   = fitz.open(stream=resp.content, filetype="pdf")
        pages = min(PAGES_PER_ARTICLE, len(doc))
        for p in range(pages):
            page = doc[p]
            mat  = fitz.Matrix(2.0, 2.0)
            pix  = page.get_pixmap(matrix=mat, colorspace=fitz.csRGB)
            fname = f"{name}_p{p:02d}.png"
            pix.save(str(IMG_DIR / fname))
            meta.append({"file_name":fname, "script":lang,
                         "family":family, "article":title, "page":p})
            saved += 1
        doc.close()
        print(f"  {name:<20} {family:<15} {pages:>5}  ✓")
    except Exception as e:
        print(f"  {name:<20} {family:<15} {'?':>5}  ✗ {str(e)[:40]}")

(INDICDLP_DIR / 'metadata.json').write_text(
    _json.dumps(meta, indent=2, ensure_ascii=False))
print(f"\nTotal saved: {saved} pages")

# ── Preview: one row per script family ──
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

imgs_all = sorted(IMG_DIR.glob('*.png'))
# Group by script family
from collections import defaultdict
by_family = defaultdict(list)
for m in meta:
    by_family[m['family']].append(IMG_DIR / m['file_name'])

families = list(by_family.keys())
MAX_COLS  = 4
n_rows    = len(families)

fig, axes = plt.subplots(n_rows, MAX_COLS, figsize=(MAX_COLS*3, n_rows*3.5))
if n_rows == 1: axes = [axes]

for row_idx, family in enumerate(families):
    row_imgs = sorted(by_family[family])[:MAX_COLS]
    for col_idx in range(MAX_COLS):
        ax = axes[row_idx][col_idx]
        if col_idx < len(row_imgs):
            ax.imshow(Image.open(str(row_imgs[col_idx])))
            ax.set_title(row_imgs[col_idx].stem[:16], fontsize=6)
        else:
            ax.set_visible(False)
        ax.axis('off')
    axes[row_idx][0].set_ylabel(family, fontsize=9, fontweight='bold', rotation=0,
                                 labelpad=80, va='center')

plt.suptitle("Real Indic Document Pages — All Scripts\n"
             "Top: Devanagari/Bengali/Tamil/Telugu  |  Bottom: Urdu/Sindhi RTL",
             fontsize=11, y=1.01)
plt.tight_layout()
prev = PROJECT_ROOT / 'output' / 'evaluation' / 'indic_all_scripts_preview.png'
prev.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(str(prev), dpi=90, bbox_inches='tight')
plt.show()
print(f"Preview saved: {prev}")
print(f"Total images ready: {len(list(IMG_DIR.glob('*.png')))}")


## Cell 13 — Zero-shot Evaluation on IndicDLP ★ SHOWS THE GAP
This shows why Phase 2 is needed: the English-trained model fails on Indic scripts.
**Expected result: mAP significantly lower than D4LA baseline (shows ~14-20% gap).**

In [ ]:
from pathlib import Path
import sys, os
PROJECT_ROOT = Path('/content/drive/MyDrive/doclayout-yolo-indic')
if not PROJECT_ROOT.parent.exists():
    from google.colab import drive
    drive.mount('/content/drive')
sys.path.insert(0, str(PROJECT_ROOT))

# ── Cell 13: Zero-shot Evaluation on ALL Indic Pages ★ SHOWS THE GAP ──
import subprocess, json, shutil
from pathlib import Path
from collections import defaultdict

subprocess.run(['pip', 'install', '-q',
    'git+https://github.com/opendatalab/DocLayout-YOLO.git'], check=True)
from doclayout_yolo import YOLOv10
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np

CKPT_LOCAL = Path('/content/doclayout_yolo_docstructbench.pt')
CKPT_DRIVE = PROJECT_ROOT / 'output' / 'checkpoints' / 'doclayout_yolo_docstructbench.pt'
IMG_DIR    = PROJECT_ROOT / 'data' / 'raw' / 'IndicDLP' / 'images' / 'test'
META_FILE  = PROJECT_ROOT / 'data' / 'raw' / 'IndicDLP' / 'metadata.json'

if not CKPT_LOCAL.exists() and CKPT_DRIVE.exists():
    shutil.copy(CKPT_DRIVE, CKPT_LOCAL)

imgs = sorted(IMG_DIR.glob('*.png'))
meta_by_file = {}
if META_FILE.exists():
    for m in json.loads(META_FILE.read_text()):
        meta_by_file[m['file_name']] = m

if not imgs:
    print("No images — run Cell 12 first.")
elif not CKPT_LOCAL.exists():
    print("Checkpoint missing — run Cell 4 first.")
else:
    model = YOLOv10(str(CKPT_LOCAL))
    COLORS = ['red','blue','green','orange','purple','brown','pink','gray','olive','cyan']

    print(f"Running inference on {len(imgs)} pages (conf threshold=0.25)...")
    print(f"{'─'*70}")

    raw_preds  = []
    log_results = []
    per_family = defaultdict(list)

    for img_path in imgs:
        res   = model.predict(source=str(img_path), imgsz=1024,
                               conf=0.25, verbose=False)[0]
        boxes = res.boxes
        confs = [float(b.conf[0]) for b in boxes]
        avg_c = round(sum(confs)/len(confs), 3) if confs else 0.0
        m     = meta_by_file.get(img_path.name, {})
        family = m.get('family', 'Unknown')

        raw_preds.append((img_path, res, family))
        log_results.append({
            "file":img_path.name, "family":family, "script":m.get('script','?'),
            "num_detections":len(boxes), "avg_conf":avg_c,
            "detections":[{"class":model.names[int(b.cls[0])],
                            "conf":round(float(b.conf[0]),3)} for b in boxes],
        })
        per_family[family].append({"n":len(boxes), "conf":avg_c})

    # ── Per-family statistics ──
    print(f"\n{'Script Family':<18} {'Pages':>5} {'Avg Det':>8} {'Avg Conf':>9} {'Gap vs English':>14}")
    print("─" * 60)
    ENG_DET  = 12.5    # from Cell 9 baseline
    ENG_CONF = 0.883
    family_stats = {}
    for fam, vals in sorted(per_family.items()):
        avg_d = sum(v['n'] for v in vals) / len(vals)
        avg_c = sum(v['conf'] for v in vals) / len(vals)
        gap_d = ENG_DET  - avg_d
        gap_c = ENG_CONF - avg_c
        family_stats[fam] = {"avg_det":avg_d, "avg_conf":avg_c,
                              "gap_det":gap_d, "gap_conf":gap_c}
        flag = "⚠️ RTL" if "RTL" in fam else ("✅" if avg_d >= 8 else "⚠️")
        print(f"  {fam:<16} {len(vals):>5} {avg_d:>8.1f} {avg_c:>8.1%}  "
              f"det:{gap_d:+5.1f}  conf:{gap_c:+5.1%}  {flag}")

    overall_det  = sum(r['num_detections'] for r in log_results) / len(log_results)
    overall_conf = sum(r['avg_conf'] for r in log_results) / len(log_results)
    print("─" * 60)
    print(f"  {'OVERALL INDIC':<16} {len(log_results):>5} {overall_det:>8.1f} "
          f"{overall_conf:>8.1%}  det:{ENG_DET-overall_det:+5.1f}  "
          f"conf:{ENG_CONF-overall_conf:+5.1%}")
    print(f"  {'ENGLISH (Cell 9)':<16} {'':>5} {ENG_DET:>8.1f} {ENG_CONF:>8.1%}")

    # ── Full grid visualisation — all pages with boxes ──
    n     = len(raw_preds)
    COLS  = 5
    ROWS  = (n + COLS - 1) // COLS
    fig, axes = plt.subplots(ROWS, COLS, figsize=(COLS*3, ROWS*3.5))
    axes_flat = list(axes.flat) if ROWS > 1 else list(axes)

    for i, (img_path, res, family) in enumerate(raw_preds):
        ax = axes_flat[i]
        ax.imshow(Image.open(str(img_path)))
        for box in res.boxes:
            x1,y1,x2,y2 = box.xyxy[0].tolist()
            cls_id = int(box.cls[0]); conf = float(box.conf[0])
            c = COLORS[cls_id % len(COLORS)]
            ax.add_patch(patches.Rectangle((x1,y1),x2-x1,y2-y1,
                lw=1.5, edgecolor=c, facecolor='none', alpha=0.85))
            ax.text(x1, max(y1-4,0),
                    f"{model.names[cls_id][:6]} {conf:.2f}",
                    color=c, fontsize=5, fontweight='bold',
                    bbox=dict(facecolor='white', alpha=0.55, pad=0))
        n_det = len(res.boxes)
        avg_c = round(sum(float(b.conf[0]) for b in res.boxes)/max(n_det,1), 2)
        title_color = 'red' if ('RTL' in family or n_det < 4) else 'black'
        ax.set_title(f"{family.replace('_RTL','⬅')[:14]}\n"
                     f"n={n_det} c={avg_c:.2f}",
                     fontsize=6.5, color=title_color)
        ax.axis('off')

    for ax in axes_flat[n:]:
        ax.axis('off')

    plt.suptitle("Zero-shot on ALL Indic pages (conf≥0.25) — DocLayout-YOLO (English-trained)\n"
                 "Red titles = RTL/failing cases  |  n=detections  c=avg_confidence",
                 fontsize=10, y=1.005)
    plt.tight_layout()
    vis = PROJECT_ROOT / 'output' / 'evaluation' / 'phase1_indic_all_detections.png'
    plt.savefig(str(vis), dpi=100, bbox_inches='tight')
    plt.show()
    print(f"\nFull detection grid saved: {vis}")

    # ── Bar chart: per-family comparison ──
    families_sorted = sorted(family_stats.items(),
                             key=lambda x: x[1]['avg_det'], reverse=True)
    fam_names = [f[0].replace('_RTL','(RTL)') for f,_ in families_sorted]
    fam_dets  = [s['avg_det']  for _,s in families_sorted]
    fam_confs = [s['avg_conf'] for _,s in families_sorted]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    colors = ['#d62728' if 'RTL' in f else '#1f77b4' for f in fam_names]

    ax1.barh(fam_names, fam_dets, color=colors, alpha=0.8)
    ax1.axvline(ENG_DET, color='green', linestyle='--', lw=2, label=f'English ({ENG_DET})')
    ax1.set_xlabel('Avg detections per page', fontsize=11)
    ax1.set_title('Detections: English vs Indic scripts', fontsize=12)
    ax1.legend(); ax1.grid(axis='x', alpha=0.3)
    for i, v in enumerate(fam_dets):
        ax1.text(v+0.2, i, f'{v:.1f}', va='center', fontsize=9)

    ax2.barh(fam_names, [c*100 for c in fam_confs], color=colors, alpha=0.8)
    ax2.axvline(ENG_CONF*100, color='green', linestyle='--', lw=2,
                label=f'English ({ENG_CONF:.0%})')
    ax2.set_xlabel('Avg confidence (%)', fontsize=11)
    ax2.set_title('Confidence: English vs Indic scripts', fontsize=12)
    ax2.legend(); ax2.grid(axis='x', alpha=0.3)
    for i, v in enumerate(fam_confs):
        ax2.text(v*100+0.5, i, f'{v:.0%}', va='center', fontsize=9)

    plt.suptitle('Model Performance Gap: English (Green line) vs Indic Scripts\n'
                 'Red bars = RTL scripts (worst failures)', fontsize=11)
    plt.tight_layout()
    bar = PROJECT_ROOT / 'output' / 'evaluation' / 'phase1_gap_chart.png'
    plt.savefig(str(bar), dpi=120, bbox_inches='tight')
    plt.show()
    print(f"Gap chart saved: {bar}")

    # ── Save ──
    out = PROJECT_ROOT / 'output' / 'evaluation' / 'phase1_indic_zeroshot.json'
    out.write_text(json.dumps({
        "phase":1,"dataset":"Wikipedia Indic (multi-script)",
        "num_images":len(log_results),
        "avg_detections_per_image":round(overall_det,2),
        "avg_confidence":round(overall_conf,3),
        "per_family_stats":family_stats,
        "per_image_results":log_results,
    }, indent=2, ensure_ascii=False))
    print(f"Results saved: {out}")


## Cell 14 — Save Phase 1 Summary Report

In [ ]:
from pathlib import Path
import sys, os
PROJECT_ROOT = Path('/content/drive/MyDrive/doclayout-yolo-indic')
if not PROJECT_ROOT.parent.exists():
    from google.colab import drive
    drive.mount('/content/drive')
sys.path.insert(0, str(PROJECT_ROOT))

# ── Cell 14: Phase 1 Summary Report ──
import json
from datetime import datetime

eval_dir = PROJECT_ROOT / 'output' / 'evaluation'

eng_file   = eval_dir / 'phase1_english_baseline.json'
indic_file = eval_dir / 'phase1_indic_zeroshot.json'

eng   = json.loads(eng_file.read_text())   if eng_file.exists()   else {}
indic = json.loads(indic_file.read_text()) if indic_file.exists() else {}

ENG_DET  = eng.get('avg_detections_per_page', 12.5)
ENG_CONF = eng.get('avg_confidence', 0.883)
IND_DET  = indic.get('avg_detections_per_image', 0)
IND_CONF = indic.get('avg_confidence', 0)

per_family = indic.get('per_family_stats', {})

# Find worst-performing family (RTL scripts)
worst_fam = min(per_family.items(), key=lambda x: x[1]['avg_det'],
                default=(None, {})) if per_family else (None, {})
worst_det  = worst_fam[1].get('avg_det',  0)
worst_conf = worst_fam[1].get('avg_conf', 0)

summary = {
    "phase":1, "completed_at":datetime.now().isoformat(),
    "model":"doclayout_yolo_docstructbench",
    "baseline_english": {"avg_det":ENG_DET, "avg_conf":ENG_CONF},
    "zero_shot_indic":  {"avg_det":IND_DET, "avg_conf":IND_CONF},
    "worst_script":     {"family":worst_fam[0], "avg_det":worst_det, "avg_conf":worst_conf},
    "gaps": {
        "detection_drop_overall":  round(ENG_DET  - IND_DET,  2),
        "confidence_drop_overall": round(ENG_CONF - IND_CONF, 4),
        "detection_drop_worst":    round(ENG_DET  - worst_det, 2) if worst_det else None,
        "confidence_drop_worst":   round(ENG_CONF - worst_conf, 4) if worst_conf else None,
    },
    "phase1_complete": eng.get('baseline_confirmed', False),
    "next_notebook":   "Phase2_SyntheticData_Colab.ipynb",
}

(eval_dir / 'phase1_summary.json').write_text(json.dumps(summary, indent=2))

W = 62
print("=" * W)
print("  PHASE 1 SUMMARY — DocLayout-YOLO Baseline vs Indic")
print("=" * W)
print(f"  {'Metric':<30} {'English':>10} {'Indic':>10} {'Gap':>8}")
print("  " + "─" * (W-2))
print(f"  {'Avg detections / page':<30} {ENG_DET:>10.1f} {IND_DET:>10.1f} {ENG_DET-IND_DET:>+8.1f}")
print(f"  {'Avg confidence':<30} {ENG_CONF:>10.1%} {IND_CONF:>10.1%} {ENG_CONF-IND_CONF:>+8.1%}")
print()
if worst_fam[0]:
    print(f"  Worst script ({worst_fam[0]}):")
    print(f"  {'  Avg detections':<30} {ENG_DET:>10.1f} {worst_det:>10.1f} {ENG_DET-worst_det:>+8.1f}")
    print(f"  {'  Avg confidence':<30} {ENG_CONF:>10.1%} {worst_conf:>10.1%} {ENG_CONF-worst_conf:>+8.1%}")
print()
if per_family:
    print(f"  {'Per-script breakdown':<30} {'Avg Det':>10} {'Avg Conf':>10}")
    print("  " + "─" * (W-2))
    for fam, s in sorted(per_family.items(), key=lambda x: x[1]['avg_det']):
        rtl = " ⬅RTL" if 'RTL' in fam else ""
        print(f"  {(fam+rtl):<30} {s['avg_det']:>10.1f} {s['avg_conf']:>10.1%}")
print()
print("=" * W)
print(f"  Baseline confirmed : {'✅ YES' if summary['phase1_complete'] else '⚠️  Run Cell 9'}")
print(f"  Phase 1 complete   : {'✅ — proceed to Phase 2' if summary['phase1_complete'] else '⚠️'}")
print("=" * W)
print()
print("  THESIS STATEMENT:")
print(f"  DocLayout-YOLO (English-trained) achieves {ENG_DET:.1f} det/page @")
print(f"  {ENG_CONF:.0%} confidence on English documents. On Indic scripts,")
print(f"  performance drops to {IND_DET:.1f} det/page @ {IND_CONF:.0%} confidence overall,")
if worst_fam[0]:
    print(f"  with worst failure on {worst_fam[0]} scripts: {worst_det:.1f} det/page")
    print(f"  @ {worst_conf:.0%} confidence — a {ENG_CONF-worst_conf:.0%} confidence drop.")
print(f"  This motivates Phase 2: training on Indic synthetic data.")


## Cell 15 — Backup Checkpoint to Drive
Ensures the checkpoint is saved persistently on Google Drive.

In [ ]:
from pathlib import Path
import sys, os
PROJECT_ROOT = Path('/content/drive/MyDrive/doclayout-yolo-indic')
if not PROJECT_ROOT.parent.exists():
    from google.colab import drive
    drive.mount('/content/drive')
sys.path.insert(0, str(PROJECT_ROOT))

# ── Cell 15: Backup Checkpoint + Verify Drive ──
import shutil

CKPT_LOCAL = Path('/content/doclayout_yolo_docstructbench.pt')
CKPT_DRIVE = PROJECT_ROOT / 'output' / 'checkpoints' / 'doclayout_yolo_docstructbench.pt'
CKPT_DRIVE.parent.mkdir(parents=True, exist_ok=True)

if CKPT_LOCAL.exists():
    shutil.copy(CKPT_LOCAL, CKPT_DRIVE)
    print(f"Checkpoint backed up  : {CKPT_DRIVE}")
    print(f"Size                  : {CKPT_DRIVE.stat().st_size/1e6:.0f} MB ✓")
elif CKPT_DRIVE.exists():
    print(f"Drive checkpoint OK   : {CKPT_DRIVE} ✓")
else:
    print("ERROR: Checkpoint not found. Re-run Cell 4.")

print("\nDrive output/ contents:")
output_dir = PROJECT_ROOT / 'output'
for f in sorted(output_dir.rglob('*')):
    if f.is_file():
        size = f.stat().st_size
        unit, val = ('MB', size/1e6) if size > 1e6 else ('KB', size/1e3)
        print(f"  {str(f.relative_to(output_dir)):<50} {val:6.1f} {unit}")


## Phase 1 Completion Checklist

After running all cells, tick these off in `Daily_Tracker.md`:

- [ ] Baseline DocLayout-YOLO reproduced → see `phase1_d4la_baseline.json`
- [ ] Baseline mAP ≥ 70% on D4LA test set
- [ ] Zero-shot mAP ≥ 75% on DocLayNet
- [ ] Zero-shot on IndicDLP shows model struggles (low detection count)
- [ ] Checkpoint backed up to Google Drive
- [ ] Results JSON files saved to `output/evaluation/`
- [ ] GitHub updated (commit results and notebook)

**If all checked → open `Phase2_SyntheticData_Colab.ipynb` to start Phase 2.**

---

### Troubleshooting

| Problem | Fix |
|---|---|
| `Model not found` | Re-run Cell 4 (download checkpoint) |
| `CUDA not available` | Runtime → Change runtime type → GPU → T4 |
| `D4LA download fails` | Download manually from GitHub, upload to Drive |
| `mAP below 70%` | Check class names in `d4la.yaml` match model classes |
| `Colab disconnects` | Run keep-alive cell from JUNE01.md in separate tab |